# M3 · Retrieval avanzado y orquestación de la normalización terminológica

Este notebook ejecuta y evalúa el retrieval del sistema RAG sobre guías clínicas. La lógica vive en
módulos `.py`; aquí solo se llaman en orden:

1. Pruebas unitarias.
2. Configuración e índice Chroma, reconstruido y verificado contra los chunks JSON.
3. Validación y resumen del gold set.
4. Prueba de humo de las técnicas de búsqueda.
5. Experimento controlado: todos los sistemas sobre las mismas consultas.
6. Análisis por tipo de consulta y decisiones de la normalización.
7. Comparación con otro reranker (opcional).
8. Integración con la generación de respuestas.

La implementación parte del laboratorio S08 (BM25, fusión RRF con k = 60 y reranker cross-encoder
mMARCO) y lo adapta al corpus del proyecto. Las rutas, los modelos y los parámetros están en
`M3/retrieval/config_retrieval.yaml`; para usar el corpus definitivo basta con cambiar `corpus: mock`
por `corpus: real` en ese archivo.

El código se descarga del repositorio del proyecto al comienzo de cada sesión, así que siempre se
ejecuta la última versión publicada en la rama `RAMA`. Con `corpus: real`, los chunks y el manifiesto
se leen de Drive, donde los deja la ingesta, y el índice se reconstruye en el disco local a partir de
esos chunks. El gold set del corpus real está en el repositorio (`M3/retrieval/data/real/`).

## 0 · Código desde el repositorio

La primera vez se clona la rama; en ejecuciones siguientes se sincroniza con la versión publicada.
La sincronización descarta los cambios hechos a los archivos dentro de Colab: los cambios de código o
de configuración deben publicarse en el repositorio. Para probar otra configuración sin publicarla,
modifíquela en el diccionario `cfg` de la sección 2 (por ejemplo, `cfg['corpus'] = 'real'`).

In [1]:
import os, subprocess

REPO_URL = 'https://github.com/luisNP21/Topicos-IA.git'
RAMA = 'dev'                        # rama con la versión a probar
RAIZ = '/content/Topicos-IA'

def git(*args, cwd=RAIZ):
    r = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} falló:\n{r.stderr.strip()}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(RAIZ, '.git')):
    git('fetch', 'origin', RAMA)
    git('checkout', '-B', RAMA, f'origin/{RAMA}')
    git('reset', '--hard', f'origin/{RAMA}')
else:
    git('clone', '--branch', RAMA, REPO_URL, RAIZ, cwd='/content')

%cd {RAIZ}
print(git('log', '-1', f'--format=Rama {RAMA} · commit %h · %ad · %s', '--date=short'))

/content/Topicos-IA
Rama dev · commit 1612fa7 · 2026-09-28 · Agregar gold set para chunks


In [2]:
%pip install -q sentence-transformers chromadb rank_bm25 groq python-dotenv PyYAML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.

In [3]:
import sys

for ruta in ['M3/retrieval', 'M3/generacion', 'M3/corpus']:
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

requeridos = ['M3/retrieval/retrieval.py', 'M3/retrieval/gold_set.py', 'M3/retrieval/config_retrieval.yaml', 'M3/retrieval/data/mock/corpus_mock.json',
              'M3/corpus/corpus_store.py', 'M3/corpus/embeddings.py', 'M3/corpus/config.yaml',
              'M3/generacion/contratos.py', 'M3/generacion/mocks.py', 'M3/generacion/generacion.py',
              'M3/generacion/orquestacion.py']
faltantes = [r for r in requeridos if not os.path.exists(r)]
if faltantes:
    raise FileNotFoundError(f'La rama {RAMA} no tiene: {faltantes}')
print('Archivos completos.')

Archivos completos.


## 1 · Pruebas unitarias

Verifican la lógica con modelos simulados, sin descargar nada.

In [4]:
!cd M3/retrieval && python -m unittest test_retrieval
!cd M3/generacion && python -m unittest test_generacion

.........Aviso: no hay umbrales calibrados en /tmp/tmpi70epy6p/umbrales.json; se usan los provisionales del YAML.
Aviso: no hay umbrales calibrados en /tmp/tmpwsvhgbjj/umbrales.json; se usan los provisionales del YAML.
..................Índice reconstruido en /tmp/tmpubroeb87/mock/chroma: 4 chunks, 1 guías.
Índice reconstruido en /tmp/tmpubroeb87/real/chroma: 4 chunks, 1 guías.
.Aviso: se reconstruye el índice de /tmp/tmp5jr3fqjj/chroma (El índice no coincide con los chunks JSON: el índice tiene 7 chunks y la carpeta de chunks 4; 3 ids del índice no están en los JSON (p. ej. ['epoc_c0', 'sepsis_c0', 'sifilis_c0']).)
Índice reconstruido en /tmp/tmp5jr3fqjj/chroma: 4 chunks, 1 guías.
.......................
----------------------------------------------------------------------
Ran 51 tests in 0.225s

OK
........[13:08:55] Piezas REALES: retrieval (M3/retrieval/config_retrieval.yaml) + normalizacion (corpus 'mock').
..    [orquestacion] falló el retrieval ('neumonia'): indice caido
    [o

## 2 · Configuración e índice

La configuración se lee de `config_retrieval.yaml`. El índice Chroma se construye siempre en el disco
local de Colab a partir de los chunks JSON del perfil, con el mismo código de la ingesta
(`M3/corpus/embeddings.py` y `corpus_store.py`). Si ya existe y coincide con los chunks, se reutiliza;
si no, se reconstruye. Después se verifica que el número, los ids y los textos del índice coincidan
con los chunks JSON y, si difieren, la celda se detiene.

Con `corpus: mock` los chunks son un corpus de prueba de 13 guías con tres secciones cada una. Con
`corpus: real` se monta Drive, donde la ingesta deja los chunks y el manifiesto; del manifiesto solo
se usan las guías que están en el índice.

In [5]:
from config_retrieval import (cargar_config, configurar_pipeline, indice_desde_config, normalizador_desde_config,
                              parametros_orquestacion, rutas)

cfg = cargar_config()
R = rutas(cfg)
print('Corpus:', cfg['corpus'], '| chunks:', R['corpus_json'], '| índice:', R['chroma_dir'], '| resultados:', R['salida'])

if cfg['corpus'] == 'real':
    from google.colab import drive
    drive.mount('/content/drive')          # los chunks y el manifiesto del corpus real están en Drive

indice = configurar_pipeline(cfg)          # reconstruye el índice si no coincide con los chunks y lo verifica

Corpus: real | chunks: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M3/data/guias_clinicas/chunks | índice: /content/chroma_guias | resultados: /content/Topicos-IA/M3/retrieval/outputs
Mounted at /content/drive


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

## 3 · Gold set

Las consultas y sus chunks relevantes se leen de las rutas del perfil activo en el YAML (`consultas`,
`anotaciones` y `gold_meta`). Antes de medir se comprueba que el gold set corresponda al índice
cargado: que existan todos los `chunk_id`, que cada chunk pertenezca a una guía relevante, que ninguna
consulta dentro del corpus quede sin chunks y que la huella del corpus coincida con la de
`gold_meta.json`. Si algo falla, la celda se detiene y dice qué comparación falló. El experimento
repite esta validación.

El gold set del corpus real lo anotó una IA simulando a un anotador humano, sin revisión humana.

In [6]:
import pandas as pd
from gold_set import cargar_gold, resumen_gold, validar_gold

gold = cargar_gold(cfg)
display(pd.DataFrame(validar_gold(indice, gold)).set_index('Comprobación'))   # se detiene si no corresponde

resumen_del_gold = resumen_gold(gold)
print('Anotador:', resumen_del_gold['anotador'] or 'sin dato')
display(pd.DataFrame(resumen_del_gold['por_tipo']).set_index('Tipo de consulta'))
display(pd.DataFrame(resumen_del_gold['por_guia']).set_index('Guía'))
display(pd.DataFrame(resumen_del_gold['por_consulta']).set_index('Consulta'))

,Resultado,Detalle
Comprobación,,
Huella del corpus,correcta,e261c2e78ef60911
Número de chunks,correcta,64
Chunks por guía,correcta,3 guías
Número de consultas,correcta,40
Chunks relevantes existen en el índice,correcta,
Chunks relevantes pertenecen a una guía relevante,correcta,
Guías relevantes existen en el índice,correcta,
Consultas dentro del corpus con chunks relevantes,correcta,
Anotaciones cubren el índice,correcta,64 chunks


Anotador: IA (Claude), simulando un anotador humano. Sin revisión humana.


,Consultas
Tipo de consulta,
Nombre exacto,3
Sigla,3
Sinónimo,3
Errata o sin tildes,3
Guías parecidas,1
Hiperespecífica,9
Fuera del corpus,18


,Consultas,Chunks relevantes distintos,Chunks de la guía,Chunks anotados de tratamiento
Guía,,,,
gpc_depresion_adultos,7,15,29,13
gpc_diabetes_mellitus_tipo_2,7,5,12,4
gpc_falla_cardiaca,8,10,23,10


,Tipo de consulta,Guía,Chunks relevantes
Consulta,,,
diabetes mellitus tipo 2,Nombre exacto,gpc_diabetes_mellitus_tipo_2,4
DM2,Sigla,gpc_diabetes_mellitus_tipo_2,4
diabetes tipo 2,Sinónimo,gpc_diabetes_mellitus_tipo_2,4
diabetis tipo 2,Errata o sin tildes,gpc_diabetes_mellitus_tipo_2,4
diabetes,Guías parecidas,gpc_diabetes_mellitus_tipo_2,4
diabetes mellitus tipo 2 con HbA1c mayor de 9 %,Hiperespecífica,gpc_diabetes_mellitus_tipo_2,1
diabetes mellitus tipo 2 en adulto mayor con fragilidad,Hiperespecífica,gpc_diabetes_mellitus_tipo_2,1
falla cardíaca,Nombre exacto,gpc_falla_cardiaca,10
insuficiencia cardíaca,Sinónimo,gpc_falla_cardiaca,10


## 4 · Prueba de humo

Se busca una consulta del gold set por cada tipo de consulta.

In [7]:
import collections
from retrieval import info_indice

info = info_indice(indice)
print(f"Índice: {info['n_chunks']} chunks, {info['n_documentos']} guías, huella {info['huella_sha256']}")
print('Secciones:', dict(collections.Counter(zip(indice.seccion_tipo, indice.seccion_origen))))

MODOS_HUMO = {'densa': 'Denso (ingenuo)', 'rerank': 'Híbrido + reranker',
              'intencion_sin_rerank': 'Tratamiento, sin reranker', 'intencion': 'Tratamiento + reranker',
              'intencion_seccion': 'Tratamiento + reranker + sección'}
humo = {}
for c in gold['consultas']:
    humo.setdefault(c['tipo'], c['consulta'])
for q in humo.values():
    print(f'\n{q}')
    for modo, nombre in MODOS_HUMO.items():
        top = indice.buscar(q, 3, modo)
        print(f'  {nombre:34}', [(f['chunk_id'].replace('mock_', ''), f['seccion_tipo'], round(f['score'], 3))
                                   for f in top])

Índice: 64 chunks, 3 guías, huella e261c2e78ef60911
Secciones: {('tratamiento', 'titulo'): 16, ('tratamiento', 'similitud'): 24, ('otra', 'titulo'): 16, ('otra', 'similitud'): 8}

diabetes mellitus tipo 2
  Denso (ingenuo)                    [('gpc_diabetes_mellitus_tipo_2_chunk3', 'tratamiento', 0.893), ('gpc_diabetes_mellitus_tipo_2_chunk0', 'tratamiento', 0.882), ('gpc_diabetes_mellitus_tipo_2_chunk1', 'tratamiento', 0.881)]
  Híbrido + reranker                 [('gpc_diabetes_mellitus_tipo_2_chunk6', 'otra', 0.829), ('gpc_diabetes_mellitus_tipo_2_chunk5', 'otra', 0.711), ('gpc_diabetes_mellitus_tipo_2_chunk10', 'tratamiento', 0.669)]
  Tratamiento, sin reranker          [('gpc_diabetes_mellitus_tipo_2_chunk1', 'tratamiento', 0.097), ('gpc_diabetes_mellitus_tipo_2_chunk3', 'tratamiento', 0.097), ('gpc_diabetes_mellitus_tipo_2_chunk0', 'tratamiento', 0.096)]
  Tratamiento + reranker             [('gpc_diabetes_mellitus_tipo_2_chunk10', 'tratamiento', 0.861), ('gpc_diabetes_mellitus_t

Cada resultado muestra el chunk, el tipo de sección y el score. Conviene revisar tres cosas: que las
técnicas con consultas de tratamiento pongan primero un chunk de sección `tratamiento`, que las
consultas fuera del corpus reciban scores bajos del reranker y cuántos chunks se clasificaron
por título y cuántos por similitud. En el corpus real, los clasificados por similitud deben revisarse
a mano en una muestra.

## 5 · Experimento controlado

| Sistema | Qué agrega |
|---|---|
| Denso (RAG ingenuo) | Línea base de S07: similitud de embeddings |
| Híbrido BM25 + denso | Búsqueda léxica fusionada con RRF |
| Híbrido + reranker | Cross-encoder sobre los 30 mejores candidatos |
| Híbrido + reranker con pregunta de tratamiento | El reranker recibe «¿Cuál es el tratamiento de…?» |
| Consultas de tratamiento, sin reranker | Tres reformulaciones con intención fusionadas con RRF |
| Consultas de tratamiento + reranker | Las reformulaciones más el reranker con la pregunta |
| … + filtro de sección | El reranker solo ordena chunks de secciones de tratamiento |
| Sistema completo | Normalización terminológica y compuerta de evidencia sobre `orquestacion.sistema_base` del YAML |
| Sistema completo + regla de siglas | Normaliza siempre las siglas |

Las dos filas de la mitad (reranker con pregunta y consultas sin reranker) separan qué aporta cada
pieza de la query transformation. Las métricas «Tratamiento» exigen que llegue el chunk de
tratamiento, no solo la guía correcta.

Los umbrales del sistema completo se calibran con validación cruzada: cada consulta se evalúa con
umbrales calibrados en los otros pliegues. En el corpus de prueba, las etiquetas de sección provienen
de los mismos títulos que usa el filtro de sección, así que ese sistema tiene ventaja por construcción.

In [8]:
import pandas as pd
from experimento_s08 import NOMBRE, TIPOS_CONSULTA, cargar_consultas, correr_desde_config, correr_experimento

detalle, resumen, calibracion = correr_desde_config(cfg, indice)

tabla = pd.DataFrame(resumen).drop(columns='Clave').set_index('Sistema')
display(tabla.T)

Sistema,Denso (RAG ingenuo),Híbrido BM25 + denso,Híbrido + reranker,Híbrido + reranker con pregunta de tratamiento,"Consultas de tratamiento, sin reranker",Consultas de tratamiento + reranker,Consultas de tratamiento + reranker + filtro de sección,Sistema completo (normalización + compuerta),Sistema completo + regla de siglas
Consultas con guía en el corpus,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000
Guía correcta en 1.er lugar,0.955,0.955,0.909,0.909,0.909,0.909,0.909,0.955,1.000
Guía correcta en el top-5,1.000,1.000,0.955,0.909,1.000,0.909,0.955,0.955,1.000
MRR de la guía,0.970,0.970,0.932,0.909,0.955,0.909,0.918,0.955,1.000
Guías relevantes en el top-5,1.000,1.000,0.955,0.909,1.000,0.909,0.955,0.955,1.000
Precisión de guía en el top-5,0.936,0.945,0.936,0.909,0.900,0.909,0.918,0.955,1.000
Tratamiento en 1.er lugar,0.045,0.136,0.273,0.636,0.182,0.682,0.727,0.727,0.773
Tratamiento en el top-5,0.409,0.409,0.864,0.909,0.682,0.909,0.909,0.955,1.000
MRR del tratamiento,0.161,0.239,0.502,0.761,0.340,0.784,0.805,0.830,0.875
Precisión de tratamiento en el top-5,0.118,0.127,0.255,0.473,0.227,0.491,0.555,0.545,0.548


In [9]:
cols = {'umbral': 'Umbral para normalizar', 'umbral_evidencia': 'Umbral de evidencia',
        'hay_zona_dudosa': 'Existe zona dudosa', 'evidencia_conservada': 'Evidencia correcta conservada'}
umbrales = pd.DataFrame([{'Calibración': f"Pliegue {p['pliegue']} (sin sus consultas)", **p} for p in calibracion['pliegues']]
                        + [{'Calibración': 'Final (todas las consultas)', **calibracion['final']}])
display(umbrales.set_index('Calibración')[list(cols)].rename(columns=cols))
print('Validación:', calibracion['validacion'])

,Umbral para normalizar,Umbral de evidencia,Existe zona dudosa,Evidencia correcta conservada
Calibración,,,,
Pliegue 0 (sin sus consultas),0.174136,0.174136,False,1.0
Pliegue 1 (sin sus consultas),0.176584,0.176584,False,1.0
Pliegue 2 (sin sus consultas),0.174136,0.115467,True,1.0
Final (todas las consultas),0.174136,0.174136,False,1.0


Validación: cruzada, 3 pliegues estratificados por tipo de consulta


Los umbrales finales son los que se configuran en el pipeline. Si varían mucho entre pliegues, la
calibración todavía depende de pocas consultas. «Existe zona dudosa» indica si el umbral de evidencia
quedó por debajo del de normalización; si no, el sistema solo distingue entre confiable y sin evidencia.

## 6 · Análisis por tipo de consulta

In [10]:
df = pd.DataFrame(detalle)
orden = [NOMBRE[f['Clave']] for f in resumen]
dentro = df[~df.fuera_de_corpus]

def por_tipo(campo):
    t = dentro.pivot_table(index='tipo', columns='sistema', values=campo, aggfunc='mean')
    return t[[c for c in orden if c in t.columns]].rename(index=TIPOS_CONSULTA).rename_axis(index='Tipo de consulta',
                                                                                           columns=None).round(2).T

print('Tratamiento en 1.er lugar, por tipo de consulta')
display(por_tipo('hit1_sec'))
print('MRR de la guía, por tipo de consulta')
display(por_tipo('rr'))

Tratamiento en 1.er lugar, por tipo de consulta


Tipo de consulta,Guías parecidas,Errata o sin tildes,Nombre exacto,Hiperespecífica,Sigla,Sinónimo
Denso (RAG ingenuo),0.0,0.00,0.00,0.11,0.00,0.00
Híbrido BM25 + denso,0.0,0.00,0.33,0.11,0.00,0.33
Híbrido + reranker,0.0,0.00,0.00,0.56,0.00,0.33
Híbrido + reranker con pregunta de tratamiento,1.0,1.00,1.00,0.56,0.00,0.67
"Consultas de tratamiento, sin reranker",0.0,0.33,0.00,0.22,0.00,0.33
Consultas de tratamiento + reranker,1.0,1.00,1.00,0.56,0.33,0.67
Consultas de tratamiento + reranker + filtro de sección,1.0,1.00,1.00,0.56,0.33,1.00
Sistema completo (normalización + compuerta),1.0,1.00,1.00,0.56,0.67,0.67
Sistema completo + regla de siglas,1.0,1.00,1.00,0.56,1.00,0.67


MRR de la guía, por tipo de consulta


Tipo de consulta,Guías parecidas,Errata o sin tildes,Nombre exacto,Hiperespecífica,Sigla,Sinónimo
Denso (RAG ingenuo),1.0,1.0,1.0,1.0,0.78,1.0
Híbrido BM25 + denso,1.0,1.0,1.0,1.0,0.78,1.0
Híbrido + reranker,1.0,1.0,1.0,1.0,0.50,1.0
Híbrido + reranker con pregunta de tratamiento,1.0,1.0,1.0,1.0,0.33,1.0
"Consultas de tratamiento, sin reranker",1.0,1.0,1.0,1.0,0.67,1.0
Consultas de tratamiento + reranker,1.0,1.0,1.0,1.0,0.33,1.0
Consultas de tratamiento + reranker + filtro de sección,1.0,1.0,1.0,1.0,0.40,1.0
Sistema completo (normalización + compuerta),1.0,1.0,1.0,1.0,0.67,1.0
Sistema completo + regla de siglas,1.0,1.0,1.0,1.0,1.00,1.0


In [11]:
piezas = ['reranker', 'reranker_pregunta', 'intencion_sin_reranker', 'intencion']
aporte = tabla.loc[[NOMBRE[p] for p in piezas], ['Tratamiento en 1.er lugar', 'MRR del tratamiento',
                                                   'Latencia (ms por consulta)']]
display(aporte)

,Tratamiento en 1.er lugar,MRR del tratamiento,Latencia (ms por consulta)
Sistema,,,
Híbrido + reranker,0.273,0.502,306.2
Híbrido + reranker con pregunta de tratamiento,0.636,0.761,309.2
"Consultas de tratamiento, sin reranker",0.182,0.340,60.3
Consultas de tratamiento + reranker,0.682,0.784,340.3


Esta tabla separa las dos piezas de la query transformation. Si «Consultas de tratamiento, sin
reranker» ya alcanza a «Consultas de tratamiento + reranker», la mejora viene de las reformulaciones y
el reranker no se gana su lugar; si queda por debajo, el reranker sí aporta. «Híbrido + reranker con
pregunta de tratamiento» mide el efecto de la pregunta sin las reformulaciones.

In [12]:
decisiones = df[df.clave == 'completo_siglas'][['consulta', 'tipo', 'tool', 'query_final', 'hit1_sec', 'rechazo',
                                                'tool_reason']]
decisiones = decisiones.assign(tipo=decisiones.tipo.map(TIPOS_CONSULTA)).rename(columns={
    'consulta': 'Consulta', 'tipo': 'Tipo de consulta', 'tool': 'Usó la normalización', 'query_final': 'Consulta final',
    'hit1_sec': 'Tratamiento en 1.er lugar', 'rechazo': 'Sin fragmentos entregados', 'tool_reason': 'Motivo'})
display(decisiones.reset_index(drop=True))

,Consulta,Tipo de consulta,Usó la normalización,Consulta final,Tratamiento en 1.er lugar,Sin fragmentos entregados,Motivo
0,diabetes mellitus tipo 2,Nombre exacto,0.0,diabetes mellitus tipo 2,1.0,0,
1,DM2,Sigla,1.0,diabetes mellitus tipo 2,1.0,0,entidad con forma de sigla (score=0.453); norm...
2,diabetes tipo 2,Sinónimo,0.0,diabetes tipo 2,1.0,0,
3,diabetis tipo 2,Errata o sin tildes,0.0,diabetis tipo 2,1.0,0,
4,diabetes,Guías parecidas,0.0,diabetes,1.0,0,
5,diabetes mellitus tipo 2 con HbA1c mayor de 9 %,Hiperespecífica,0.0,diabetes mellitus tipo 2 con HbA1c mayor de 9 %,1.0,0,
6,diabetes mellitus tipo 2 en adulto mayor con f...,Hiperespecífica,0.0,diabetes mellitus tipo 2 en adulto mayor con f...,0.0,0,
7,falla cardíaca,Nombre exacto,0.0,falla cardíaca,1.0,0,
8,insuficiencia cardíaca,Sinónimo,1.0,falla cardíaca,1.0,0,score=0.174 < umbral=0.177; normalizada vía SN...
9,ICC,Sigla,1.0,falla cardíaca,1.0,0,entidad con forma de sigla (score=0.103); norm...


## 7 · Otro reranker (opcional)

El reranker alternativo del YAML (`BAAI/bge-reranker-v2-m3`) es multilingüe y más grande. Conviene ejecutarlo con GPU
(Entorno de ejecución › Cambiar tipo de entorno de ejecución › T4).

In [13]:
indice_bge = indice_desde_config(cfg, modelo_reranker=cfg['modelos']['reranker_alternativo'])
_, resumen_bge, _ = correr_experimento(indice_bge, cargar_consultas(R['consultas']), k=cfg['retrieval']['k'],
                                      claves=('ingenuo', 'reranker', 'intencion'))
comparacion = pd.concat([
    tabla.loc[[NOMBRE['reranker'], NOMBRE['intencion']]].rename(index=lambda s: s + ' (mMARCO)'),
    pd.DataFrame(resumen_bge).drop(columns='Clave').set_index('Sistema')
      .loc[[NOMBRE['reranker'], NOMBRE['intencion']]].rename(index=lambda s: s + ' (bge-m3)'),
])
display(comparacion[['Guía correcta en 1.er lugar', 'Tratamiento en 1.er lugar', 'MRR del tratamiento',
                     'Latencia (ms por consulta)']])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

,Guía correcta en 1.er lugar,Tratamiento en 1.er lugar,MRR del tratamiento,Latencia (ms por consulta)
Sistema,,,,
Híbrido + reranker (mMARCO),0.909,0.273,0.502,306.2
Consultas de tratamiento + reranker (mMARCO),0.909,0.682,0.784,340.3
Híbrido + reranker (bge-m3),0.955,0.364,0.545,2542.7
Consultas de tratamiento + reranker (bge-m3),0.955,0.818,0.880,2677.3


## 8 · Integración con la generación

Se ejecuta la cadena completa para tres entidades: orquestación, retrieval y generación de la
respuesta. La generación recibe como parámetro la función que llama al LLM; aquí se usa una función
simulada que devuelve un texto fijo, para verificar el recorrido de los datos sin consumir la cuota de
la API. Para obtener respuestas reales se reemplaza por `GroqGenerator()` del módulo `generacion`. La
normalización es la simulada del perfil activo (`mapa_normalizacion` en el YAML); con
`mapa_normalizacion: null` se usa la herramienta real, que necesita `BIOPORTAL_API_KEY`.

In [14]:
from retrieval import pregunta_intencion, retrieve_advanced
from orquestacion import resolver_query
from generacion import generar_respuesta

parametros = parametros_orquestacion(cfg)       # umbrales calibrados (o provisionales si aún no hay)
normalizar = normalizador_desde_config(cfg)     # mapa simulado del YAML, o la herramienta real si es null

def generador_simulado(system, user):
    return 'Respuesta simulada con la fuente citada.'

entidades = (['neumonía adquirida en la comunidad', 'HTA', 'amiloidosis'] if cfg['corpus'] == 'mock'
             else ['diabetes mellitus tipo 2', 'DM2', 'amiloidosis'])
for entidad in entidades:
    q = resolver_query(entidad, retrieve_fn=retrieve_advanced, normalizar_fn=normalizar, **parametros)
    r = generar_respuesta(q['query_final'], q['fragments'], generar_fn=generador_simulado,
                          pregunta=pregunta_intencion(q['query_final']))
    print(f"\n{entidad}")
    print('  normalización:', 'sí' if q['tool_invoked'] else 'no', '|', q['tool_reason'] or 'score suficiente')
    print('  fragmentos:', [(f['chunk_id'], f['seccion_tipo'], f['score']) for f in q['fragments']])
    print('  respuesta:', r['answer'], '| sin evidencia' if r['fallback_used'] else '')


diabetes mellitus tipo 2
  normalización: no | score suficiente
  fragmentos: [('gpc_diabetes_mellitus_tipo_2_chunk10', 'tratamiento', 0.86144), ('gpc_diabetes_mellitus_tipo_2_chunk6', 'otra', 0.773524), ('gpc_diabetes_mellitus_tipo_2_chunk7', 'tratamiento', 0.772969), ('gpc_diabetes_mellitus_tipo_2_chunk1', 'tratamiento', 0.759757), ('gpc_diabetes_mellitus_tipo_2_chunk8', 'tratamiento', 0.730449)]
  respuesta: Respuesta simulada con la fuente citada. 

DM2
  normalización: sí | entidad con forma de sigla (score=0.453); normalizada vía SNOMED CT (mock) (score 0.453 -> 0.861)
  fragmentos: [('gpc_diabetes_mellitus_tipo_2_chunk10', 'tratamiento', 0.86144), ('gpc_diabetes_mellitus_tipo_2_chunk6', 'otra', 0.773524), ('gpc_diabetes_mellitus_tipo_2_chunk7', 'tratamiento', 0.772969), ('gpc_diabetes_mellitus_tipo_2_chunk1', 'tratamiento', 0.759757), ('gpc_diabetes_mellitus_tipo_2_chunk8', 'tratamiento', 0.730449)]
  respuesta: Respuesta simulada con la fuente citada. 

amiloidosis
  normaliza

## 9 · Descargar los resultados

Los archivos generados en Colab se pierden al cerrar la sesión. Esta celda descarga la carpeta de
resultados comprimida, para revisarla o publicarla en el repositorio.

In [15]:
import shutil
from google.colab import files

archivo = shutil.make_archive('/content/resultados_retrieval', 'zip', R['salida'])
files.download(archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>